# Objetivo 3 — ResNet-18 (Linea Base)
Fine-tuning completo desde ImageNet.  
**Segun PPI:** capa fc reemplazada por `Linear(512, n_classes)` directamente.  
2 neuronas para Chest X-Ray, 3 para IQ-OTH/NCCD.

## 0 · Instalacion

In [1]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','torch','torchvision','--index-url','https://download.pytorch.org/whl/cu128'],check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn','matplotlib','seaborn','tqdm'],check=False)
import torch; print('PyTorch:',torch.__version__)


PyTorch: 2.11.0+cu128


## 1 · Imports y seed

In [2]:
import os,random
from pathlib import Path
import numpy as np
import torch,torch.nn as nn
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18,ResNet18_Weights
from torch.utils.data import DataLoader,WeightedRandomSampler
SEED=42
random.seed(SEED);np.random.seed(SEED)
torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',DEVICE)


Device: cuda


## 2 · DataLoaders

In [3]:
BASE_DIR=Path(os.getcwd())
CHEST_OUT=BASE_DIR/'etl_output'/'chest_xray'
LUNG_OUT=BASE_DIR/'etl_output'/'lung_cancer'
IMG_SIZE=(128,128);MEAN=[0.485,0.456,0.406];STD=[0.229,0.224,0.225]
tfm_tr=T.Compose([T.Resize(IMG_SIZE),T.Grayscale(num_output_channels=3),T.RandomRotation(10),T.RandomHorizontalFlip(),T.ColorJitter(brightness=0.15),T.RandomAffine(degrees=0,scale=(0.90,1.10)),T.ToTensor(),T.Normalize(MEAN,STD)])
tfm_ev=T.Compose([T.Resize(IMG_SIZE),T.Grayscale(num_output_channels=3),T.ToTensor(),T.Normalize(MEAN,STD)])
def make_loaders(root,weighted=False,bs=32):
    root=Path(root)
    ds_tr=ImageFolder(root/'train',transform=tfm_tr)
    ds_va=ImageFolder(root/'val',transform=tfm_ev)
    ds_te=ImageFolder(root/'test',transform=tfm_ev)
    kw=dict(num_workers=2,pin_memory=True)
    if weighted:
        tgts=torch.tensor(ds_tr.targets)
        sw=(1.0/torch.bincount(tgts).float())[tgts]
        ltr=DataLoader(ds_tr,batch_size=bs,sampler=WeightedRandomSampler(sw,len(sw),True),**kw)
    else:
        ltr=DataLoader(ds_tr,batch_size=bs,shuffle=True,**kw)
    return ltr,DataLoader(ds_va,batch_size=bs,shuffle=False,**kw),DataLoader(ds_te,batch_size=bs,shuffle=False,**kw),ds_tr.class_to_idx
print('DataLoaders definidos.')


DataLoaders definidos.


## 3 · Arquitectura ResNet-18 — segun PPI
La capa `fc` original (`512 → 1000`) se reemplaza por `Linear(512, n_classes)` directamente,  
tal como especifica el PPI: *'capa densa con el numero de neuronas correspondiente al numero de clases'*.

In [4]:
def build_resnet18(n_classes:int) -> nn.Module:
    """
    ResNet-18 preentrenado en ImageNet, fine-tuning completo.
    fc: Linear(512, n_classes)  —  segun PPI (sin bottleneck intermedio).
    """
    model=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    model.fc=nn.Linear(model.fc.in_features, n_classes)  # 512 -> n_classes
    return model

with torch.no_grad():
    m2=build_resnet18(2); m3=build_resnet18(3)
    x=torch.randn(2,3,128,128)
    print('ResNet-18 output (n=2):',m2(x).shape)
    print('ResNet-18 output (n=3):',m3(x).shape)
    p=sum(p.numel() for p in m2.parameters() if p.requires_grad)
    print(f'Parametros entrenables: {p:,}')
    print('\nCapa fc:')
    print(m2.fc)


ResNet-18 output (n=2): torch.Size([2, 2])
ResNet-18 output (n=3): torch.Size([2, 3])
Parametros entrenables: 11,177,538

Capa fc:
Linear(in_features=512, out_features=2, bias=True)


## 4 · Diagrama

In [5]:
print('Arquitectura ResNet-18 (segun PPI):')
print()
print('  Conv1 -> BN -> ReLU -> MaxPool')
print('  Layer1 (2 BasicBlocks)')
print('  Layer2 (2 BasicBlocks)')
print('  Layer3 (2 BasicBlocks)')
print('  Layer4 (2 BasicBlocks)')
print('  AdaptiveAvgPool -> flatten(512)')
print('  Linear(512, n_classes)  <-- fc adaptada')
print()
print('Chest X-Ray: n_classes=2  (NORMAL, PNEUMONIA)')
print('Lung Cancer: n_classes=3  (Benign, Malignant, Normal)')


Arquitectura ResNet-18 (segun PPI):

  Conv1 -> BN -> ReLU -> MaxPool
  Layer1 (2 BasicBlocks)
  Layer2 (2 BasicBlocks)
  Layer3 (2 BasicBlocks)
  Layer4 (2 BasicBlocks)
  AdaptiveAvgPool -> flatten(512)
  Linear(512, n_classes)  <-- fc adaptada

Chest X-Ray: n_classes=2  (NORMAL, PNEUMONIA)
Lung Cancer: n_classes=3  (Benign, Malignant, Normal)
